# Enterprise BI Engineering Demo Notebook

This notebook supports the repo's trustworthy Power BI story. It keeps the experience notebook-friendly while showing how BI developers can inspect metadata, compare auth paths, and build small internal tools around Power BI artifacts.

## What this demo covers
- Compare delegated and service principal auth in one practical flow
- List Power BI workspaces, datasets, and reports
- Run a small DAX query through the REST API as a validation probe
- Connect the API flow back to PBIP/PBIR/TMDL, source control, and AI-assisted tooling

## Prerequisites
1. Install packages from `requirements.txt`.
2. Copy `.env.example` to `.env` and fill in tenant, app, and workspace details.
3. Make sure the Power BI tenant settings and dataset permissions required for your chosen auth mode are already configured.
4. The primary documented speaker path is the delegated notebook. This all-in-one notebook remains useful for workshop and comparison flows.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
scripts_path = repo_root / 'scripts'
if str(scripts_path) not in sys.path:
    sys.path.append(str(scripts_path))

from config_loader import load_config
from list_workspaces import list_workspaces
from list_datasets import list_datasets
from execute_dax_query import execute_dax_query, DEFAULT_QUERY

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## Step 1 — Load configuration
The scripts and notebook use the same configuration model so the notebook stays simple while the code remains reusable.

In [ ]:
config = load_config()
config

## Step 2 — Choose authentication mode
Use `service_principal` for app-only automation. In this legacy notebook, use `delegated_user` when you want the notebook to run as the signed-in user context. The primary documented repo flow uses `AUTH_MODE=delegated` in the split notebooks.

In [ ]:
AUTH_MODE = os.getenv('PBI_AUTH_MODE', 'service_principal')
AUTH_MODE

## Step 3 — List workspaces
This step shows the first visible difference between auth modes: the workspace list reflects the access context of the caller.

In [ ]:
workspaces_df = list_workspaces(auth_mode=AUTH_MODE, top=50)
workspaces_df

## Step 4 — Pick a workspace
You can either set `PBI_WORKSPACE_ID` in `.env` or select the first workspace for a quick demo.

In [ ]:
workspace_id = config.workspace_id or (workspaces_df.iloc[0]['id'] if not workspaces_df.empty else None)
workspace_id

## Step 5 — List datasets
This step helps the audience see that the notebook is just a friendly interface over standard REST operations.

In [ ]:
datasets_df = list_datasets(workspace_id=workspace_id, auth_mode=AUTH_MODE)
datasets_df

## Step 6 — Pick a dataset and define a DAX query
The default query returns a compact regional summary that is easy to read on screen during a live presentation.

In [ ]:
dataset_id = config.dataset_id or (datasets_df.iloc[0]['id'] if not datasets_df.empty else None)
dax_query = DEFAULT_QUERY
print(dax_query)

## Step 7 — Execute the DAX query
If your dataset uses RLS and you are demonstrating delegated auth, the results should align with the signed-in user's access. For a dual-auth demo with the exact same notebook flow, keep the main demo dataset free of RLS.

In [ ]:
query_df = execute_dax_query(
    workspace_id=workspace_id,
    dataset_id=dataset_id,
    dax_query=dax_query,
    auth_mode=AUTH_MODE,
)
query_df

## Step 8 — Optional chart
A simple bar chart makes the output feel more like a polished demo and less like a raw API response.

In [ ]:
if not query_df.empty and 'Dim Region[Region]' in query_df.columns and '[Total Sales]' in query_df.columns:
    ax = query_df.plot(kind='bar', x='Dim Region[Region]', y='[Total Sales]', legend=False, figsize=(8, 4))
    ax.set_title('Total Sales by Region')
    ax.set_xlabel('Region')
    ax.set_ylabel('Total Sales')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('Expected columns not found. Displaying table only.')

## Final summary
- The notebook is the presentation-friendly entry point.
- The scripts remain reusable for technical teams and CI/CD scenarios.
- Service principal is best for unattended automation.
- Delegated auth is best when you want the run to reflect an actual user context.